# DBSCAN Experiments (Thesis-aligned)

Notebook ini fokus pada alur: 1) impor dan konfigurasi, 2) k-distance plot untuk menentukan eps, 3) eksperimen parameter (eps & min_samples) menggunakan silhouette (sample), 4) fit final DBSCAN dan simpan model, 5) evaluasi cluster.

In [ ]:
# Part 1 — Imports & configuration
import numpy as np
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score
import gc

# CONFIG: set input embeddings file (one-file)
INPUT_FILE = Path('combined_embeddings.npy')  # change if needed
RANDOM_STATE = 42
SAMPLE_FOR_METRICS = 50000  # reduce if memory limited
KNN_NEIGHBORS = 4  # for k-distance plot (k = min_samples)

In [ ]:
# Part 2 — Load embeddings and k-distance plot to estimate eps
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Input embeddings not found: {INPUT_FILE}')
emb = np.load(INPUT_FILE)
print('Loaded embeddings shape:', emb.shape)
# compute nearest-neighbors distances (k-distance)
nn = NearestNeighbors(n_neighbors=KNN_NEIGHBORS, n_jobs=-1)
nn.fit(emb)
distances, _ = nn.kneighbors(emb)
# distances[:, -1] is the distance to k-th neighbor
k_dist = np.sort(distances[:, -1])
# plot k-distance curve
plt.figure(figsize=(8,4))
plt.plot(k_dist)
plt.xlabel('Points sorted by k-distance')
plt.ylabel(f'k-distance (k={KNN_NEIGHBORS})')
plt.title('k-distance plot — look for elbow to choose eps')
plt.tight_layout()
plt.show()

In [ ]:
# Part 3 — Parameter search over eps and min_samples (uses a sample for silhouette)
# WARNING: DBSCAN can be slow for large datasets; we sample for metric computation
n = emb.shape[0]
if n > SAMPLE_FOR_METRICS:
    rng = np.random.RandomState(RANDOM_STATE)
    sample_idx = rng.choice(n, SAMPLE_FOR_METRICS, replace=False)
    emb_sample = emb[sample_idx]
else:
    emb_sample = emb

eps_values = np.linspace(np.percentile(k_dist, 50), np.percentile(k_dist, 95), 8)
min_samples_values = [3, 5, 8, 12]
results = []

for eps in eps_values:
    for ms in min_samples_values:
        print(f'Testing eps={eps:.4g}, min_samples={ms}')
        dbs = DBSCAN(eps=float(eps), min_samples=int(ms), n_jobs=-1)
        labels = dbs.fit_predict(emb_sample)
        # compute number of clusters (exclude noise label -1)
        unique_labels = set(labels) - {-1}
        n_clusters = len(unique_labels)
        sil = -1
        if n_clusters > 1:
            try:
                sil = silhouette_score(emb_sample, labels)
            except Exception:
                sil = -1
        results.append({'eps': float(eps), 'min_samples': int(ms), 'n_clusters': n_clusters, 'silhouette': float(sil)})

# show results sorted by silhouette
import pandas as pd
df_res = pd.DataFrame(results)
print(df_res.sort_values('silhouette', ascending=False).head(10))

In [ ]:
# Part 4 — Fit final DBSCAN on full data and save model+labels
# Set chosen parameters based on Part 3 results
CHOSEN_EPS = float(df_res.sort_values('silhouette', ascending=False).iloc[0]['eps'])
CHOSEN_MIN_SAMPLES = int(df_res.sort_values('silhouette', ascending=False).iloc[0]['min_samples'])
print('Fitting DBSCAN with eps=', CHOSEN_EPS, 'min_samples=', CHOSEN_MIN_SAMPLES)
model = DBSCAN(eps=CHOSEN_EPS, min_samples=CHOSEN_MIN_SAMPLES, n_jobs=-1)
labels_full = model.fit_predict(emb)
# Save labels and model (scikit-learn DBSCAN stores core_sample_indices_ and components_)
joblib.dump(model, 'dbscan_model.pkl')
np.save('dbscan_labels.npy', labels_full)
print('Saved dbscan_model.pkl and dbscan_labels.npy')

In [ ]:
# Part 5 — Evaluation and cluster analysis
from collections import Counter
counts = Counter(labels_full)
print('Label counts (including noise -1):')
for lbl, cnt in sorted(counts.items()):
    print(f' - {lbl}: {cnt}')

# silhouette on sample or full if small
n = emb.shape[0]
if n > SAMPLE_FOR_METRICS:
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.choice(n, SAMPLE_FOR_METRICS, replace=False)
    lbls_s = labels_full[idx]
    emb_s = emb[idx]
else:
    emb_s = emb
    lbls_s = labels_full
if len(set(lbls_s) - {-1}) > 1:
    sil = silhouette_score(emb_s, lbls_s)
    try:
        dbi = davies_bouldin_score(emb_s, lbls_s)
    except Exception as e:
        dbi = None
    print('Silhouette (sample/full):', sil)
    print('Davies-Bouldin (sample/full):', dbi)
else:
    print('Not enough clusters (excluding noise) to compute silhouette/DB index')

## Notes & Next Steps
- Notebook ini fokus pada satu sumber input `combined_embeddings.npy`. Jika file Anda berbeda, ganti `INPUT_FILE`.
- DBSCAN bisa sensitif terhadap `eps` dan `min_samples`; gunakan hasil Part 3 untuk memilih parameter terbaik.
- Untuk menjalankan notebook non-interaktif (PowerShell), jalankan:

```
jupyter nbconvert --to notebook --execute dbscan\dbscan_experiments.ipynb --output dbscan_experiments_run.ipynb
```

Beritahu saya jika Anda ingin saya jalankan notebook atau sesuaikan parameter default.